# Feature Engineering

In [3]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv(r"..\data\raw\online_retail_II.csv")

print(df.shape)

df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
df_clean = df.copy()

In [ ]:
# Fix data types (Convert InvoiceDate)

df_clean['InvoiceDate'] = pd.to_datetime(
    df_clean['InvoiceDate']
)

df_clean['InvoiceDate'].dtype

dtype('<M8[us]')

In [ ]:
# Remove missing Cutomer IDs

print("Before:", df_clean.shape)

df_clean.dropna(
    subset=['Customer ID'],
    inplace=True
)

print("After:", df_clean.shape)

Before: (1067371, 8)
After: (824364, 8)


In [7]:
# Remove missing descriptions

df_clean.dropna(
    subset=['Description'],
    inplace=True
)

In [ ]:
# Remove duplicates

print("Duplicates:", df_clean.duplicated().sum())

df_clean.drop_duplicates(
    inplace=True
)

print("Duplicates after:", df_clean.duplicated().sum())

Duplicates: 26479
Duplicates after: 0


In [ ]:
# Investigate negative quantities

negative_qty = df_clean[
    df_clean['Quantity'] < 0
]

negative_qty.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia


In [ ]:
# Remove invalid prices

df_clean[df_clean['Price'] <= 0].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658.0,United Kingdom
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231.0,United Kingdom
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108.0,United Kingdom
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108.0,United Kingdom


In [14]:
df_clean = df_clean[
    df_clean['Price'] > 0
]

In [15]:
# Create revenue feature

df_clean['Revenue'] = (
    df_clean['Quantity']
    * df_clean['Price']
)

In [16]:
# Convert customer ID

df_clean['Customer ID'] = (
    df_clean['Customer ID'].astype(int).astype(str)
)

In [17]:
# Create date features

df_clean['Year'] = (
    df_clean['InvoiceDate'].dt.year
)

df_clean['Month'] = (
    df_clean['InvoiceDate'].dt.month
)

df_clean['Day'] = (
    df_clean['InvoiceDate'].dt.day
)

df_clean['Quarter'] = (
    df_clean['InvoiceDate'].dt.quarter
)

df_clean['DAyOfWeek'] = (
    df_clean['InvoiceDate'].dt.day_name()
)

df_clean['Week'] = (
    df_clean['InvoiceDate'].dt.isocalendar().week
)

In [18]:
df_clean.shape

(797815, 15)

In [19]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 797815 entries, 0 to 1067370
Data columns (total 15 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      797815 non-null  str           
 1   StockCode    797815 non-null  str           
 2   Description  797815 non-null  str           
 3   Quantity     797815 non-null  int64         
 4   InvoiceDate  797815 non-null  datetime64[us]
 5   Price        797815 non-null  float64       
 6   Customer ID  797815 non-null  str           
 7   Country      797815 non-null  str           
 8   Revenue      797815 non-null  float64       
 9   Year         797815 non-null  int32         
 10  Month        797815 non-null  int32         
 11  Day          797815 non-null  int32         
 12  Quarter      797815 non-null  int32         
 13  DAyOfWeek    797815 non-null  str           
 14  Week         797815 non-null  UInt32        
dtypes: UInt32(1), datetime64[us](1), float64(2), int3

In [20]:
df_clean.isnull().sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
Revenue        0
Year           0
Month          0
Day            0
Quarter        0
DAyOfWeek      0
Week           0
dtype: int64

In [21]:
df_clean.duplicated().sum()

np.int64(0)

In [22]:
# Save processed dataset

df_clean.to_csv(
    r"..\data\processed\online_retail_clean.csv",
    index=False
)

1. Converted the InvoiceDate column from object datatype to datetime format to enable time-based analysis.

2. Removed records with missing Customer IDs because customer-level analytics such as RFM analysis, segmentation, and CLV prediction require unique customer identification.

3. Removed records with missing product descriptions due to their negligible proportion and limited analytical value.

4. Removed duplicate transactions to prevent inflation of revenue, order count, and customer lifetime value metrics.

5. Excluded transactions with negative quantities as they represented returns and cancellations, which could distort customer purchase behavior analysis.

6. Removed transactions with zero or negative prices to eliminate invalid or refund-related records.

7. Converted Customer ID from float to string because it is an identifier rather than a numerical variable.

8. Created a new Revenue feature by multiplying Quantity and Price, which serves as a key business KPI.

9. Engineered multiple date-based features including Year, Month, Day, Quarter, Week, and DayOfWeek to support temporal analysis and downstream modeling.

10. Performed a final data quality check to ensure the cleaned dataset contained no missing values or duplicate records.

11. Saved the cleaned and feature-engineered dataset for subsequent stages, including RFM analysis, customer segmentation, churn prediction, and CLV modeling.